In [1]:
!pip install torchaudio transformers gtts sounddevice scipy pydub

In [4]:
# Install dependencies in Colab
!pip install torchaudio transformers gtts pydub scipy
!apt-get update && apt-get install -y ffmpeg

import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow logging

import torchaudio
from transformers import pipeline, WhisperForConditionalGeneration, WhisperProcessor
from gtts import gTTS
import scipy.io.wavfile as wav
import pydub
from IPython.display import Audio
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Initialize models once to avoid reloading
try:
    sentiment_classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
    whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")
    whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
except Exception as e:
    print(f"Error loading models: {e}")
    exit(1)

# Step 1: Check for input audio or use placeholder
def get_input_audio(filename="input.wav"):
    if os.path.exists(filename):
        print(f"Found {filename}. Proceeding with transcription.")
        return filename
    else:
        print(f"No {filename} found. Using placeholder text for transcription.")
        return None

# Step 2: Transcribe audio using Whisper
def transcribe_audio(filename="input.wav"):
    try:
        if filename is None:
            return "I feel tired and ignored."  # Placeholder text
        # Load audio
        waveform, sample_rate = torchaudio.load(filename)
        if sample_rate != 16000:
            resampler = torchaudio.transforms.Resample(sample_rate, 16000)
            waveform = resampler(waveform)

        # Process audio for Whisper
        inputs = whisper_processor(waveform.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        predicted_ids = whisper_model.generate(inputs["input_features"])
        transcription = whisper_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        return transcription if transcription else "No speech detected."
    except Exception as e:
        print(f"Error transcribing audio: {e}")
        return "Error in transcription."

# Step 3: Analyze sentiment
def get_sentiment(text):
    try:
        result = sentiment_classifier(text)[0]
        # Normalize label to uppercase for consistency
        label = result['label'].upper()
        score = result['score']
        return label, score
    except Exception as e:
        print(f"Error analyzing sentiment: {e}")
        return "NEUTRAL", 0.0

# Step 4: Generate feedback
def generate_response(sentiment):
    if sentiment == "NEGATIVE":
        return "I'm sorry you're feeling this way. You're not alone, and I'm here to support you."
    elif sentiment == "POSITIVE":
        return "That's awesome! Keep shining!"
    else:
        return "Thanks for sharing. Let's take a moment to reflect together."

# Step 5: Speak feedback
def speak_feedback(text, lang='en'):
    try:
        tts = gTTS(text=text, lang=lang)
        filename = "feedback.mp3"
        tts.save(filename)
        # Display audio in Colab
        display(Audio(filename, autoplay=True))
        # Clean up
        os.remove(filename)
    except Exception as e:
        print(f"Error generating feedback audio: {e}")

# Run the whole flow
def ai_sentiment_coach():
    audio_file = get_input_audio()

    text = transcribe_audio(audio_file)
    print(f"User said: {text}")

    sentiment, score = get_sentiment(text)
    print(f"Detected Sentiment: {sentiment} ({score:.2f})")

    response = generate_response(sentiment)
    print(f"Coach says: {response}")

    speak_feedback(response)

if __name__ == "__main__":
    try:
        ai_sentiment_coach()
    except KeyboardInterrupt:
        print("\nStopped by user.")
    finally:
        # Clean up any leftover files
        for file in ["input.wav", "feedback.mp3"]:
            if os.path.exists(file):
                os.remove(file)

Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,715 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,909 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2,901

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/3.75k [00:00<?, ?B/s]

No input.wav found. Using placeholder text for transcription.
User said: I feel tired and ignored.
Detected Sentiment: NEGATIVE (1.00)
Coach says: I'm sorry you're feeling this way. You're not alone, and I'm here to support you.


In [8]:
# Install kagglehub if not already installed
!pip install kagglehub

import kagglehub
import os
import random
import glob
from IPython.display import Audio
import torchaudio
from gtts import gTTS

# Ensure models are initialized from the first cell
if 'sentiment_classifier' not in globals() or 'whisper_processor' not in globals() or 'whisper_model' not in globals():
    raise RuntimeError("Models not initialized. Please run the first cell to initialize sentiment_classifier, whisper_processor, and whisper_model.")

# Download Audio Emotions dataset using kagglehub
print("Downloading Audio Emotions dataset...")
dataset_path = kagglehub.dataset_download("uldisvalainis/audio-emotions")
print("Path to dataset files:", dataset_path)

# Debug: List all .wav files in the dataset
print("Listing all .wav files in dataset:")
wav_files_list = glob.glob(os.path.join(dataset_path, "**/*.wav"), recursive=True)
print(wav_files_list)
if not wav_files_list:
    print("WARNING: No .wav files found in dataset. Ensure the dataset downloaded correctly.")

# Step 1: Get a random audio file from the dataset
def get_input_audio():
    # Refresh list of .wav files to ensure randomness
    wav_files = glob.glob(os.path.join(dataset_path, "**/*.wav"), recursive=True)
    if wav_files:
        selected_file = random.choice(wav_files)
        print(f"Selected dataset file: {selected_file}")
        return selected_file
    else:
        print(f"No dataset .wav files found. Using placeholder text.")
        return None

# Step 2: Transcribe audio using Whisper
def transcribe_audio(filename=None):
    try:
        if filename is None:
            return "I feel tired and ignored."  # Placeholder text
        # Load audio
        waveform, sample_rate = torchaudio.load(filename)
        if sample_rate != 16000:
            resampler = torchaudio.transforms.Resample(sample_rate, 16000)
            waveform = resampler(waveform)

        # Process audio for Whisper
        inputs = whisper_processor(waveform.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        predicted_ids = whisper_model.generate(inputs["input_features"])
        transcription = whisper_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        return transcription if transcription else "No speech detected."
    except Exception as e:
        print(f"Error transcribing audio: {e}")
        return "Error in transcription."

# Step 3: Analyze sentiment
def get_sentiment(text):
    try:
        result = sentiment_classifier(text)[0]
        # Normalize label to uppercase for consistency
        label = result['label'].upper()
        score = result['score']
        return label, score
    except Exception as e:
        print(f"Error analyzing sentiment: {e}")
        return "NEUTRAL", 0.0

# Step 4: Generate feedback
def generate_response(sentiment):
    if sentiment == "NEGATIVE":
        return "I'm sorry you're feeling this way. You're not alone, and I'm here to support you."
    elif sentiment == "POSITIVE":
        return "That's awesome! Keep shining!"
    else:
        return "Thanks for sharing. Let's take a moment to reflect together."

# Step 5: Speak feedback
def speak_feedback(text, lang='en'):
    try:
        tts = gTTS(text=text, lang=lang)
        filename = "feedback.mp3"
        tts.save(filename)
        # Display audio in Colab
        display(Audio(filename, autoplay=True))
        # Clean up
        os.remove(filename)
    except Exception as e:
        print(f"Error generating feedback audio: {e}")

# Run the whole flow with a new random voice
def ai_sentiment_coach():
    audio_file = get_input_audio()

    text = transcribe_audio(audio_file)
    print(f"User said: {text}")

    sentiment, score = get_sentiment(text)
    print(f"Detected Sentiment: {sentiment} ({score:.2f})")

    response = generate_response(sentiment)
    print(f"Coach says: {response}")

    speak_feedback(response)

if __name__ == "__main__":
    try:
        ai_sentiment_coach()
    except KeyboardInterrupt:
        print("\nStopped by user.")
    finally:
        # Clean up any leftover files
        for file in ["feedback.mp3"]:
            if os.path.exists(file):
                os.remove(file)

100%|██████████| 1.12G/1.12G [00:11<00:00, 107MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/uldisvalainis/audio-emotions/versions/1
Listing all .wav files in dataset:
['/root/.cache/kagglehub/datasets/uldisvalainis/audio-emotions/versions/1/Emotions/Neutral/1036_IEO_NEU_XX.wav', '/root/.cache/kagglehub/datasets/uldisvalainis/audio-emotions/versions/1/Emotions/Neutral/03-01-01-01-02-01-13.wav', '/root/.cache/kagglehub/datasets/uldisvalainis/audio-emotions/versions/1/Emotions/Neutral/1068_ITH_NEU_XX.wav', '/root/.cache/kagglehub/datasets/uldisvalainis/audio-emotions/versions/1/Emotions/Neutral/1025_WSI_NEU_XX.wav', '/root/.cache/kagglehub/datasets/uldisvalainis/audio-emotions/versions/1/Emotions/Neutral/1071_ITS_NEU_XX.wav', '/root/.cache/kagglehub/datasets/uldisvalainis/audio-emotions/versions/1/Emotions/Neutral/1077_ITS_NEU_XX.wav', '/root/.cache/kagglehub/datasets/uldisvalainis/audio-emotions/versions/1/Emotions/Neutral/1060_ITH_NEU_XX.wav', '/root/.cache/kagglehub/datasets/uldisvalainis/audio-emotions/versions/1/Emotion

Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


User said:  The surface is slick.
Detected Sentiment: POSITIVE (1.00)
Coach says: That's awesome! Keep shining!
